# ByteTrack + RF-DETR Object Tracking Test

Windows 환경에서 ByteTrack 트래커를 테스트하는 노트북입니다.

## 1. GPU 확인

In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4080 SUPER


## 2. 의존성 설치

In [5]:
# inference-gpu와 trackers 설치
!pip install -q inference-gpu trackers supervision

^C


## 3. 샘플 비디오 다운로드

In [6]:
import urllib.request
from pathlib import Path

videos = [
    "https://storage.googleapis.com/com-roboflow-marketing/supervision/video-examples/bikes-1280x720-1.mp4",
    "https://storage.googleapis.com/com-roboflow-marketing/supervision/video-examples/bikes-1280x720-2.mp4",
    "https://storage.googleapis.com/com-roboflow-marketing/supervision/video-examples/skiers-1280x720-5.mp4",
]

for url in videos:
    filename = Path(url).name
    if not Path(filename).exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"  Done!")
    else:
        print(f"{filename} already exists, skipping.")

bikes-1280x720-1.mp4 already exists, skipping.
bikes-1280x720-2.mp4 already exists, skipping.
skiers-1280x720-5.mp4 already exists, skipping.


## 4. 모델 및 트래커 초기화

In [ ]:
# RF -DETR 사용

from inference import get_model
from trackers import ByteTrackTracker

model = get_model("rfdetr-medium")
tracker = ByteTrackTracker()


print("Model and tracker initialized!")

ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.To suppress this warning, set CORE_MODEL_SAM_ENABLED to False.
ModelDependencyMissing: Your `inference` configuration does not support SAM3 model. Install SAM3 dependencies and set CORE_MODEL_SAM3_ENABLED to True.
ModelDependencyMissing: Your `inference` configuration does not support Gaze Detection model. Use pip install 'inference[gaze]' to install missing requirements.To suppress this warning, set CORE_MODEL_GAZE_ENABLED to False.


Model and tracker initialized!


In [6]:
# YOLO26 사용
from ultralytics import YOLO 
from trackers import ByteTrackTracker
import supervision as sv

model = YOLO(r"C:\Users\User\Desktop\smart-farm\runs\yolo26_merged_tomato\weights\best.pt")
tracker = ByteTrackTracker()


## 5. Annotator 설정

In [7]:
import supervision as sv

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

print("Annotators configured!")

Annotators configured!


## 6. Detection + Tracking 실행

In [8]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

SOURCE_VIDEO_PATH = "rgb_(1).mp4"
TARGET_VIDEO_PATH = "rgb_(1)-result.mp4"

def callback(frame, i):
    # RF -DETR 사용 시 
    #result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    #detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)

    # YOLO26 사용 시 
    results = model(frame, conf=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_ultralytics(results).with_nms(threshold=NMS_THRESHOLD)
    
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    
    labels = [str(tid) for tid in detections.tracker_id] if detections.tracker_id is not None else []
    annotated_image = label_annotator.annotate(annotated_image, detections, labels)

    return annotated_image

tracker.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback
)

print(f"\nTracking complete! Result saved to: {TARGET_VIDEO_PATH}")


0: 384x640 5 tomatos, 39.4ms
Speed: 7.9ms preprocess, 39.4ms inference, 11.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 6.1ms
Speed: 1.4ms preprocess, 6.1ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 5.5ms
Speed: 1.9ms preprocess, 5.5ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 6.8ms
Speed: 1.0ms preprocess, 6.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 5.8ms
Speed: 0.8ms preprocess, 5.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 6.0ms
Speed: 1.1ms preprocess, 6.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 5.9ms
Speed: 2.1ms preprocess, 5.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 tomatos, 5.7ms
Speed: 1.3ms preprocess, 5.7ms inference, 0.4ms postprocess per image at shape (1, 3, 

## 7. 결과 확인

In [11]:
from IPython.display import Video

# 비디오가 큰 경우 압축 (ffmpeg 필요)
import subprocess
import shutil

if shutil.which("ffmpeg"):
    TARGET_VIDEO_COMPRESSED = "bikes-1280x720-1-result-compressed.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", TARGET_VIDEO_PATH,
        "-vcodec", "libx264", "-crf", "28",
        TARGET_VIDEO_COMPRESSED
    ])
    Video(TARGET_VIDEO_COMPRESSED, embed=True, width=800)
else:
    print(f"ffmpeg not found. Please open the video file directly: {TARGET_VIDEO_PATH}")
    Video(TARGET_VIDEO_PATH, embed=True, width=800)

## 8. 두 번째 비디오 테스트

In [12]:
SOURCE_VIDEO_PATH = "bikes-1280x720-2.mp4"
TARGET_VIDEO_PATH = "bikes-1280x720-2-result.mp4"

tracker.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback
)

print(f"\nTracking complete! Result saved to: {TARGET_VIDEO_PATH}")


Tracking complete! Result saved to: bikes-1280x720-2-result.mp4


## 9. Camera Motion Compensation (이동 카메라용)

In [13]:
from inference import get_model
from trackers import ByteTrackTracker, MotionEstimator, MotionAwareTraceAnnotator

PERSON_CLASS_ID = 0

model_large = get_model("rfdetr-large")
tracker_motion = ByteTrackTracker(minimum_consecutive_frames=3)
motion_estimator = MotionEstimator(
    max_points=500,
    min_distance=10,
    quality_level=0.001,
    ransac_reproj_threshold=1.0,
)

motion_aware_trace_annotator = MotionAwareTraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100,
)

print("Motion compensation initialized!")

Motion compensation initialized!


In [14]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

SOURCE_VIDEO_PATH = "skiers-1280x720-5.mp4"
TARGET_VIDEO_PATH = "skiers-1280x720-5-result.mp4"

def callback_motion(frame, i):
    coord_transform = motion_estimator.update(frame)

    result = model_large.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker_motion.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = motion_aware_trace_annotator.annotate(
        annotated_image, detections, coord_transform=coord_transform)
    
    labels = [str(tid) for tid in detections.tracker_id] if detections.tracker_id is not None else []
    annotated_image = label_annotator.annotate(annotated_image, detections, labels)

    return annotated_image

tracker_motion.reset()
motion_estimator.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback_motion
)

print(f"\nTracking with motion compensation complete! Result saved to: {TARGET_VIDEO_PATH}")


Tracking with motion compensation complete! Result saved to: skiers-1280x720-5-result.mp4


## Done!

ByteTrack + RF-DETR 조합으로 객체 추적을 완료했습니다.

참고:
- [trackers Documentation](https://roboflow.github.io/trackers/)
- [GitHub](https://github.com/roboflow/trackers)